# TransHLA — Phase 1: Collect Data

Tái hiện bài báo **TransHLA: a Hybrid Transformer model for HLA-presented epitope detection**
(Lu, Wang et al., *GigaScience* 2025, DOI 10.1093/gigascience/giaf008).

Notebook này **chỉ thu thập và chuẩn hoá dữ liệu** — không cần GPU.
Đầu ra được ghi vào Google Drive để Phase 2 và Phase 3 dùng lại.

---

### Lưu ý quan trọng về quy trình sinh negative samples

Trong bài báo, negative samples được sinh bằng cách:
1. BLAST (DIAMOND) các peptide dương tính ngược lại **NCBI nr database** để tìm protein nguồn,
2. cắt ngẫu nhiên các đoạn không chồng lấn từ các protein đó,
3. khử dư thừa bằng CD-HIT ở ngưỡng 0.8.

**Bước này KHÔNG chạy được trên Colab**: database `nr` nặng ~400 GB (sau khi index DIAMOND vẫn
~200 GB) và cần vài chục GB RAM. Vì vậy mặc định notebook dùng **bộ dữ liệu train/val/test
mà tác giả đã công bố** (GitHub + GigaDB) — đây là cách tái hiện đúng và khả thi nhất.

Nếu bạn vẫn muốn tự sinh negative, phần `OPTION B` ở cuối notebook mô tả pipeline đầy đủ
để chạy trên một máy chủ có đủ dung lượng (không phải Colab).

## 0. Đồng bộ code từ GitHub

Code được sửa trong **VS Code** trên máy, `git push` lên GitHub, ô dưới đây kéo bản mới nhất về Colab.
Repo công khai nên chạy được ngay, không cần token.

> Ô này **ghi đè** mọi sửa đổi phía Colab. Nguồn sự thật là GitHub.


In [ ]:
# === Đồng bộ code từ GitHub — chạy ô này đầu tiên, mỗi phiên Colab ===
REPO = "github.com/Tienanh209/TransHLA_repro.git"
BRANCH = "main"

import os, sys, pathlib, subprocess

def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True)

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN") or ""
except Exception:
    pass

url = f"https://{token + '@' if token else ''}{REPO}"
WORK = pathlib.Path("/content/TransHLA_repro")

if (WORK / ".git").exists():
    _sh("git", "-C", str(WORK), "remote", "set-url", "origin", url)
    _sh("git", "-C", str(WORK), "fetch", "origin", BRANCH)
    _sh("git", "-C", str(WORK), "reset", "--hard", f"origin/{BRANCH}")
else:
    r = _sh("git", "clone", "--branch", BRANCH, url, str(WORK))
    if r.returncode:
        raise SystemExit((r.stderr.replace(token, "***") if token else r.stderr))

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code tại:", WORK, "|", _sh("git", "-C", str(WORK), "log", "-1", "--oneline").stdout.strip())


## 0. Cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

# Thư mục làm việc trên Drive (giữ nguyên qua các phiên Colab)
PROJ = pathlib.Path('/content/drive/MyDrive/s1156052/EpitopePrediction')
DATA = PROJ / 'data'          # dữ liệu đã chuẩn hoá  (Phase 1 -> Phase 2/3)
RAW  = PROJ / 'raw'           # dữ liệu thô tải về
CKPT = PROJ / 'checkpoints'   # Phase 3
RES  = PROJ / 'results'       # Phase 3
for p in (PROJ, DATA, RAW, CKPT, RES):
    p.mkdir(parents=True, exist_ok=True)

# Độ dài peptide theo bài báo
LEN_I  = (8, 14)    # HLA class I
LEN_II = (13, 21)   # HLA class II

print(PROJ)

In [ ]:
!pip -q install pandas pyarrow biopython tqdm

## 1. Tải source code + dataset của tác giả

Repo `SkywalkerLuke/TransHLA` chứa code huấn luyện (`model_train_test/`), script inference
(`TransHLA_I.py`, `TransHLA_II.py`) và các file dữ liệu. Bản lưu trữ chính thức kèm dữ liệu
đầy đủ nằm ở **GigaDB** (trích dẫn [55] trong bài báo).

In [ ]:
%cd /content
!rm -rf TransHLA
!git clone --depth 1 https://github.com/SkywalkerLuke/TransHLA.git
%cd /content/TransHLA
!ls -la

In [ ]:
# Liệt kê toàn bộ file dữ liệu có trong repo để biết chính xác tên/định dạng
import subprocess
print(subprocess.run(
    ['bash','-lc',
     "find /content/TransHLA -type f \\( -name '*.csv' -o -name '*.tsv' -o -name '*.txt' "
     "-o -name '*.parquet' -o -name '*.npy' -o -name '*.zip' -o -name '*.gz' \\) "
     "-printf '%10s  %p\\n' | sort -k2"],
    capture_output=True, text=True).stdout)

In [ ]:
# Xem README để đối chiếu quy ước cột / tên file (phòng khi tác giả cập nhật repo)
from IPython.display import Markdown, display
import glob
for f in glob.glob('/content/TransHLA/README*'):
    display(Markdown(open(f, encoding='utf-8', errors='ignore').read()[:6000]))

### 1b. (Tuỳ chọn) Tải bản dữ liệu đầy đủ từ GigaDB

Nếu repo GitHub không kèm đủ file train/val/test, lấy bản lưu trữ GigaDB.
Mở trang DOI của bài báo → mục *Data Availability* → link GigaDB, rồi điền `GIGADB_FILES` bên dưới.
Dữ liệu GigaDB thường tải được qua FTP công khai.

In [ ]:
# Bỏ comment và điền đúng đường dẫn FTP nếu cần
# GIGADB_FILES = [
#     'ftp://parrot.genomics.cn/gigadb/pub/10.5524/102001_103000/102XXX/train_test_data.zip',
# ]
# for url in GIGADB_FILES:
#     !wget -c -P {RAW} "{url}"
# !cd {RAW} && for z in *.zip; do unzip -o "$z"; done

## 2. Nạp và chuẩn hoá dữ liệu

Mục tiêu: tạo ra 1 schema thống nhất cho mọi tập dữ liệu

| cột | ý nghĩa |
|---|---|
| `peptide` | chuỗi acid amin, viết hoa, chỉ gồm 20 acid amin chuẩn |
| `label` | 1 = epitope được trình diện, 0 = peptide thường |
| `hla_class` | `I` hoặc `II` |
| `allele` | allele HLA (nếu có) — dùng để vẽ Hình 2 |
| `source` | tên database gốc (IEDB / CEDAR / VDJdb / ImmuneCode / dbPepNeo2 / NEPDB) |
| `split` | `train` / `val` / `test` / `external` |

In [ ]:
import pandas as pd, numpy as np, re, glob

AA = set('ACDEFGHIKLMNPQRSTVWY')
STD_COLS = ['peptide', 'label', 'hla_class', 'allele', 'source', 'split']

def clean_peptides(df, pep_col='peptide'):
    # Lọc peptide: viết hoa, bỏ ký tự lạ, bỏ trùng lặp
    df = df.copy()
    df[pep_col] = df[pep_col].astype(str).str.strip().str.upper()
    ok = df[pep_col].map(lambda s: len(s) > 0 and set(s) <= AA)
    dropped = (~ok).sum()
    if dropped:
        print(f'  bỏ {dropped} peptide chứa ký tự không chuẩn')
    return df[ok]

def filter_length(df, hla_class, pep_col='peptide'):
    lo, hi = LEN_I if hla_class == 'I' else LEN_II
    n0 = len(df)
    df = df[df[pep_col].str.len().between(lo, hi)]
    print(f'  lọc độ dài [{lo},{hi}]: {n0} -> {len(df)}')
    return df

def finalize(df, hla_class, source, split, allele_col=None):
    df = clean_peptides(df)
    df = filter_length(df, hla_class)
    out = pd.DataFrame({
        'peptide':   df['peptide'].values,
        'label':     df['label'].astype(int).values,
        'hla_class': hla_class,
        'allele':    df[allele_col].values if allele_col and allele_col in df else '',
        'source':    source,
        'split':     split,
    })
    # Khử trùng lặp peptide trong cùng một split (giữ bản ghi đầu tiên)
    n0 = len(out)
    out = out.drop_duplicates(subset=['peptide', 'label'])
    print(f'  khử trùng lặp: {n0} -> {len(out)}')
    return out[STD_COLS]

In [ ]:
def sniff(path, n=3):
    # In vài dòng đầu + tên cột của một file để xác định schema
    try:
        df = pd.read_csv(path, nrows=n, sep=None, engine='python')
        print(f'--- {path}\n    cols = {list(df.columns)}')
        display(df.head(n))
        return df.columns.tolist()
    except Exception as e:
        print(f'--- {path}  [không đọc được: {e}]')
        return None

candidates = sorted(glob.glob('/content/TransHLA/**/*.csv', recursive=True)) + \
             sorted(glob.glob('/content/TransHLA/**/*.tsv', recursive=True))
for c in candidates[:30]:
    sniff(c)

### 2a. Ánh xạ file → split

**Chạy ô `sniff` ở trên trước**, xem tên cột thực tế, rồi chỉnh `FILE_MAP` cho khớp.
Mẫu dưới đây giả định repo có các file dạng `train_I.csv`, `val_I.csv`, `test_I.csv`, …
với cột `peptide` và `label`. Nếu tên cột khác (ví dụ `sequence`, `y`), đổi trong `RENAME`.

In [ ]:
RENAME = {  # tên cột trong file gốc -> tên chuẩn
    'sequence': 'peptide', 'Sequence': 'peptide', 'Peptide': 'peptide',
    'epitope': 'peptide', 'Epitope': 'peptide', 'pep': 'peptide',
    'y': 'label', 'Label': 'label', 'target': 'label', 'class': 'label',
    'Allele': 'allele', 'HLA': 'allele', 'mhc': 'allele', 'MHC': 'allele',
}

# (đường_dẫn_glob, hla_class, split, source)
FILE_MAP = [
    ('/content/TransHLA/**/*train*I.csv',   'I',  'train', 'IEDB'),
    ('/content/TransHLA/**/*val*I.csv',     'I',  'val',   'IEDB'),
    ('/content/TransHLA/**/*test*I.csv',    'I',  'test',  'IEDB'),
    ('/content/TransHLA/**/*train*II.csv',  'II', 'train', 'IEDB'),
    ('/content/TransHLA/**/*val*II.csv',    'II', 'val',   'IEDB'),
    ('/content/TransHLA/**/*test*II.csv',   'II', 'test',  'IEDB'),
]

def load_mapped(file_map):
    frames = []
    for pattern, hla_class, split, source in file_map:
        paths = sorted(glob.glob(pattern, recursive=True))
        if not paths:
            print(f'[!] không tìm thấy file nào khớp: {pattern}')
            continue
        for p in paths:
            print(f'\n[{split}/{hla_class}] {p}')
            df = pd.read_csv(p, sep=None, engine='python')
            df = df.rename(columns=RENAME)
            if 'peptide' not in df.columns:
                print('    [!] không có cột peptide — bỏ qua'); continue
            if 'label' not in df.columns:
                print('    [!] không có cột label — bỏ qua'); continue
            frames.append(finalize(df, hla_class, source, split,
                                   allele_col='allele' if 'allele' in df else None))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STD_COLS)

main = load_mapped(FILE_MAP)
print('\nTổng:', len(main))
main.groupby(['hla_class','split','label']).size()

### 2b. Phương án dự phòng: tự chia train/val/test 7:1:2

Nếu repo chỉ cung cấp một file gộp (positive + negative) mà không chia sẵn,
dùng ô này để chia theo đúng tỉ lệ bài báo (7:1:2), phân tầng theo nhãn.

In [ ]:
from sklearn.model_selection import train_test_split

def split_721(df, seed=42):
    tr, tmp = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=seed)
    val, te = train_test_split(tmp, test_size=2/3, stratify=tmp['label'], random_state=seed)
    tr, val, te = tr.copy(), val.copy(), te.copy()
    tr['split'], val['split'], te['split'] = 'train', 'val', 'test'
    return pd.concat([tr, val, te], ignore_index=True)

# Ví dụ dùng khi cần:
# pooled = finalize(pd.read_csv('/content/TransHLA/data/HLA_I_all.csv'), 'I', 'IEDB', 'train')
# main = split_721(pooled)

## 3. Các tập external test (case study)

Bài báo dùng **CEDAR, VDJdb, ImmuneCode, dbPepNeo2.0** làm external test và **NEPDB**
cho bài toán neoepitope. Tiêu chí lọc nêu trong mục *Data collection*:

* CEDAR: `MHC Ligand`, `Linear Peptide`, `Host = Human`, `Outcome = Positive`
* VDJdb: `Species = Human`
* ImmuneCode: dùng toàn bộ dữ liệu thô
* dbPepNeo2.0: neoantigen mức tin cậy **HC**
* NEPDB: `Response = positive`

Số lượng sau xử lý trong bài báo: **21.387** epitope HLA-I và **3.580** epitope HLA-II,
kèm lượng chuỗi ngẫu nhiên âm tính bằng đúng số dương tính.

In [ ]:
EXT_MAP = [
    ('/content/TransHLA/**/*CEDAR*',      None, 'external', 'CEDAR'),
    ('/content/TransHLA/**/*VDJdb*',      None, 'external', 'VDJdb'),
    ('/content/TransHLA/**/*mmune*ode*',  None, 'external', 'ImmuneCode'),
    ('/content/TransHLA/**/*epNeo*',      None, 'external', 'dbPepNeo2'),
    ('/content/TransHLA/**/*NEP*',        None, 'external', 'NEPDB'),
]

def load_external(pattern, source):
    frames = []
    for p in sorted(glob.glob(pattern, recursive=True)):
        if not p.lower().endswith(('.csv', '.tsv', '.txt')):
            continue
        try:
            df = pd.read_csv(p, sep=None, engine='python').rename(columns=RENAME)
        except Exception as e:
            print(f'  [!] {p}: {e}'); continue
        if 'peptide' not in df.columns:
            continue
        if 'label' not in df.columns:
            df['label'] = 1           # các DB này chỉ chứa mẫu dương tính
        print(f'  {p}: {len(df)} dòng')
        # tách theo lớp HLA dựa trên độ dài peptide
        for hc in ('I', 'II'):
            sub = finalize(df, hc, source, 'external',
                           allele_col='allele' if 'allele' in df else None)
            if len(sub):
                frames.append(sub)
    return frames

ext_frames = []
for pattern, _, split, source in EXT_MAP:
    print(f'\n=== {source} ===')
    ext_frames += load_external(pattern, source)

external = pd.concat(ext_frames, ignore_index=True) if ext_frames else pd.DataFrame(columns=STD_COLS)
print('\nExternal tổng:', len(external))
external.groupby(['source','hla_class']).size()

### 3b. Sinh negative ngẫu nhiên cho external test

Bài báo: *"we added random sequences in the same quantity as the identified persistent epitopes"*.
Ở đây sinh chuỗi ngẫu nhiên theo **tần suất acid amin của chính tập dương tính** (giữ cùng phân bố
độ dài), và loại bỏ mọi chuỗi trùng với bất kỳ peptide dương tính nào trong toàn bộ dữ liệu.

In [ ]:
rng = np.random.default_rng(42)

def random_negatives(pos_df, forbidden:set, source):
    # Sinh số negative = số positive, khớp phân bố độ dài và tần suất acid amin
    aa_counts = pd.Series(list(''.join(pos_df['peptide']))).value_counts()
    aa_list   = aa_counts.index.to_numpy()
    aa_prob   = (aa_counts / aa_counts.sum()).to_numpy()
    lengths   = pos_df['peptide'].str.len().to_numpy()

    out, seen = [], set()
    for L in lengths:
        for _ in range(100):
            s = ''.join(rng.choice(aa_list, size=L, p=aa_prob))
            if s not in forbidden and s not in seen:
                seen.add(s); out.append(s); break
    return pd.DataFrame({
        'peptide': out, 'label': 0,
        'hla_class': pos_df['hla_class'].iloc[0],
        'allele': '', 'source': source + '_random', 'split': 'external',
    })[STD_COLS]

all_pos = set(pd.concat([main, external])['peptide'][
    pd.concat([main, external])['label'] == 1])

neg_frames = []
for (src, hc), g in external[external.label == 1].groupby(['source', 'hla_class']):
    if len(g) == 0: continue
    neg_frames.append(random_negatives(g, all_pos, src))
    print(f'{src}/{hc}: sinh {len(neg_frames[-1])} negative')

if neg_frames:
    external = pd.concat([external] + neg_frames, ignore_index=True)
print('External sau khi thêm negative:', len(external))

## 4. Kiểm tra chất lượng dữ liệu

In [ ]:
full = pd.concat([main, external], ignore_index=True)

print('=== Số mẫu theo lớp / split / nhãn (so với Bảng 1 của bài báo) ===')
display(full.groupby(['hla_class','split','label']).size().unstack(fill_value=0))

print('\nBài báo — Bảng 1:')
print('  HLA-I : 459,442 dương  /  459,442 âm   (chia 7:1:2)')
print('  HLA-II: 312,245 dương  /  312,245 âm   (chia 7:1:2)')
print('  Test IEDB: 92,347 epitope HLA-I, 65,105 epitope HLA-II, 157,879 chuỗi ngẫu nhiên')

In [ ]:
# Rò rỉ dữ liệu: peptide xuất hiện ở cả train và test?
for hc in ('I','II'):
    d = full[full.hla_class == hc]
    tr = set(d[d.split=='train'].peptide)
    te = set(d[d.split=='test'].peptide)
    va = set(d[d.split=='val'].peptide)
    print(f'HLA-{hc}:  train∩test = {len(tr&te)},  train∩val = {len(tr&va)}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, hc in zip(axes, ('I','II')):
    d = full[(full.hla_class==hc) & (full.split.isin(['train','val','test']))]
    for lab, name in [(1,'epitope'), (0,'non-epitope')]:
        v = d[d.label==lab].peptide.str.len()
        ax.hist(v, bins=range(d.peptide.str.len().min(), d.peptide.str.len().max()+2),
                alpha=.55, label=name)
    ax.set_title(f'HLA-{hc}: phân bố độ dài peptide'); ax.set_xlabel('độ dài'); ax.legend()
plt.tight_layout(); plt.show()

### Hình 2 — phân bố allele HLA theo split

Bài báo kiểm tra tính hợp lý của cách chia dữ liệu bằng cách so sánh phân bố tần suất allele
giữa train/val/test (Hình 2 cho HLA-I, Hình S5 cho HLA-II): **258 allele HLA-I** và
**227 allele HLA-II**. Ô dưới chỉ chạy được nếu dữ liệu có cột `allele`.

In [ ]:
def plot_allele_dist(df, hla_class, gene_prefix, top=50):
    d = df[(df.hla_class==hla_class) & (df.label==1) & (df.allele.astype(str).str.len()>0)]
    d = d[d.allele.str.contains(gene_prefix, na=False)]
    if d.empty:
        print(f'không có dữ liệu allele cho {gene_prefix}'); return
    order = d.allele.value_counts().head(top).index
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.2), sharey=False)
    for ax, sp in zip(axes, ['train','val','test']):
        c = d[d.split==sp].allele.value_counts().reindex(order, fill_value=0)
        ax.bar(range(len(c)), c.values); ax.set_title(f'{gene_prefix} — {sp}')
        ax.set_xticks([])
    plt.tight_layout(); plt.show()

if full.allele.astype(str).str.len().gt(0).any():
    n_I  = full[(full.hla_class=='I')  & (full.label==1)].allele.nunique()
    n_II = full[(full.hla_class=='II') & (full.label==1)].allele.nunique()
    print(f'Số allele duy nhất — HLA-I: {n_I} (bài báo: 258), HLA-II: {n_II} (bài báo: 227)')
    for g in ['HLA-A','HLA-B','HLA-C']:   plot_allele_dist(full, 'I',  g)
    for g in ['HLA-DR','HLA-DQ','HLA-DP']: plot_allele_dist(full, 'II', g)
else:
    print('Dữ liệu không có cột allele — bỏ qua phần tái hiện Hình 2.')

## 5. Lưu dữ liệu đã chuẩn hoá

In [ ]:
for hc in ('I','II'):
    d = full[full.hla_class == hc].reset_index(drop=True)
    out = DATA / f'hla{hc}_all.parquet'
    d.to_parquet(out, index=False)
    print(f'{out}  —  {len(d):,} dòng')

(DATA / 'MANIFEST.txt').write_text(
    'TransHLA reproduction — Phase 1 output\n'
    f'rows total: {len(full)}\n' +
    full.groupby(["hla_class","split","label"]).size().to_string()
)
print('\nXong Phase 1. Chuyển sang 02_process_data.ipynb')

---

## OPTION B — Tự sinh negative samples từ đầu (không chạy trên Colab)

Nếu bạn muốn dựng lại hoàn toàn bộ dữ liệu từ IEDB thay vì dùng bộ của tác giả, đây là
pipeline tương ứng với mô tả trong bài báo. Cần một máy có **≥ 500 GB đĩa** và **≥ 64 GB RAM**;
riêng bước DIAMOND mất khoảng **1–2 ngày**.

```bash
# 1. Tải positive từ IEDB
#    https://www.iedb.org/database_export_v3.php  ->  mhc_ligand_full.zip  (~2 GB)
#    Lọc: MHC Ligand / Linear Peptide / Host = Human / Outcome = Positive
#    Độ dài: 8–14 (HLA-I), 13–21 (HLA-II)

# 2. Tải NCBI nr (~400 GB khi giải nén)
wget -c https://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/nr.gz
diamond makedb --in nr.gz -d nr --threads 32          # ~3-5 giờ

# 3. BLAST peptide dương tính ngược lại nr để tìm protein nguồn
diamond blastp -d nr.dmnd -q positives.fasta -o hits.tsv \
    --ultra-sensitive --id 100 --query-cover 100 --threads 32   # ~1-2 ngày

# 4. Lấy các protein nguồn, cắt đoạn ngẫu nhiên KHÔNG chồng lấn peptide dương tính
#    (cùng phân bố độ dài với tập dương tính)

# 5. Khử dư thừa ở ngưỡng 0.8
cd-hit -i negatives_raw.fasta -o negatives.fasta -c 0.8 -n 5 -M 32000 -T 32

# 6. Cân bằng số lượng dương/âm, chia 7:1:2
```

Peptide dương tính nào **không** khớp bất kỳ protein nào trong nr sẽ bị loại khỏi tập dương
tính (theo đúng mô tả của bài báo).